# EmotiveAD · V01 estable para Colab
Ejecuta de arriba abajo con GPU (Entorno de ejecución → Cambiar tipo → GPU).
Entrada: `datos/Locutores`, `datos/videos prueba mp4` y JSON existentes en `datos/Prueba 2 VDMS (XTTS)/1. Input`.
No se llaman APIs pagadas para reutilizar tus JSON. El pipeline histórico de creación de JSON queda archivado;
para nuevas descripciones automáticas usa V02 (Qwen local).

**Opciones:** 1 XTTS, 2 Spanish-F5, 3 ambos (60 audios para 10 videos × 3 locutores).
Se usan los mismos textos y referencias para cada motor. Los pesos se descargan en la primera ejecución.
Este notebook se validó estructuralmente; la inferencia GPU debe verificarse en tu sesión de Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, json, shutil
# Sube la carpeta V01 Original completa a Drive; edita SOLO esta ruta.
V01 = Path('/content/drive/MyDrive/EmotiveAD/V01 Original') # @param {type:"string"}
DATA = V01 / 'datos'
OUT = V01 / 'outputs'
assert (V01/'emotivead').is_dir(), 'No se encuentra V01 Original; corrige V01.'
assert (DATA/'Locutores').is_dir(), 'Faltan los datos. Sube V01 Original/datos completo.'
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
UV = shutil.which('uv')
subprocess.run([UV,'venv','--python','3.11','/content/emotive-core'],check=True)
PY = '/content/emotive-core/bin/python'
subprocess.run([UV,'pip','install','--python',PY,'-r',str(V01/'requirements.txt')],check=True)
subprocess.run([UV,'pip','check','--python',PY],check=True)
os.environ['PYTHONPATH'] = str(V01)
print('Entorno aislado listo; no se sustituyó NumPy ni CUDA del kernel de Colab.')

In [ ]:
# @title Configuración experimental
OPTION = 3 # @param [1,2,3] {type:"raw"}
NARRATORS = ['Andrea','Sergio','Sebastian']
MIX = True # @param {type:"boolean"}
ENGINES = {1:['xtts'],2:['spanish_f5'],3:['xtts','spanish_f5']}[OPTION]
# XTTS usa CPML. Lee https://huggingface.co/coqui/XTTS-v2/blob/main/LICENSE.txt
ACCEPT_XTTS_LICENSE = False # @param {type:"boolean"}
if 'xtts' in ENGINES and not ACCEPT_XTTS_LICENSE:
    raise ValueError('Para XTTS, lee la licencia y marca ACCEPT_XTTS_LICENSE si la aceptas; o elige otro motor.')
if ACCEPT_XTTS_LICENSE: os.environ['COQUI_TOS_AGREED']='1'
subprocess.run([PY,str(V01/'setup_envs.py'),'--engines',*ENGINES,'--target','/content/emotive-engines'],check=True)
ENV = '/content/emotive-engines/env.json'

In [ ]:
# Validación de entradas sin sintetizar: revisar projects.json antes de continuar.
subprocess.run([PY,str(V01/'run_pipeline.py'),'--root',str(DATA),'--output',str(OUT),
                '--option',str(OPTION),'--prepare-only'],check=True)
projects=json.loads((OUT/'projects.json').read_text())
for p in projects:
    s=p['segments'][0]
    print(p['video_id'], round(p['duration'],2), s['emotion'], s['text'])

In [ ]:
# Síntesis y mezcla. Puede tardar; cada motor se descarga y carga de forma secuencial.
cmd=[PY,str(V01/'run_pipeline.py'),'--root',str(DATA),'--output',str(OUT),
     '--option',str(OPTION),'--env',ENV,'--narrators',*NARRATORS]
if MIX: cmd.append('--mix')
subprocess.run(cmd,check=True)
print('Revisa mix_report.json por video: needs_edit indica que una voz no cabe.')

In [ ]:
# Evaluación sobre WAV RAW, antes de atempo, ducking o normalización de escucha.
REPORT = OUT/'evaluacion_raw'
subprocess.run([PY,str(V01/'evaluate.py'),'--root',str(DATA),'--manifest',str(OUT/'audio_manifest.json'),
                '--stage','raw','--output',str(REPORT)],check=True)
from IPython.display import display, Image
display(Image(filename=str(REPORT/'01_dispersion_referencia.png')))
display(Image(filename=str(REPORT/'03_distribuciones.png')))

In [ ]:
# Empaquetar únicamente resultados nuevos (no las voces de referencia ni credenciales).
from google.colab import files
archive=shutil.make_archive('/content/EmotiveAD_resultados','zip',OUT)
files.download(archive)

## Comprobación final
Escucha las tres voces, verifica el texto y revisa `failures.json`, `mix_report.json` y `extraction_errors.json`.
Los audios y manifiestos permanecen en Drive si se desconecta Colab. Reejecutar comprueba hashes antes de reutilizar síntesis.
Spanish-F5 puede transcribir la referencia localmente si no hay `.txt` junto al WAV. Para reproducibilidad,
añade su transcripción exacta y revisada; no uses como referencia el texto que quieres generar.

La transferencia emocional no está garantizada por la etiqueta de emoción. El condicionamiento usa
la grabación emocional del locutor; confirma el resultado con escucha y evaluación perceptual.